# Amazon Product & Pricing Analysis
### Data Analyst Portfolio Project

**Objective:** Analyse Amazon product listings to understand pricing, discounts, ratings, review engagement, and category-level patterns.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

**Business questions**
- Which product categories contain the most listings?
- Which categories have higher average ratings and review engagement?
- How do actual prices, discounted prices, and discount percentages vary?
- Is there an observable relationship between discount, rating, and review count?
- What data-quality issues should be considered before making recommendations?

> **Dataset:** This notebook is designed for an Amazon product dataset with fields such as `product_id`, `product_name`, `category`, `discounted_price`, `actual_price`, `discount_percentage`, `rating`, and `rating_count`. The dataset was not bundled with the notebook, so add your CSV file before running the analysis.

## 1. Import libraries and set up the notebook

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

# Update this path to where your CSV file is saved.
DATA_PATH = Path("amazon.csv")

## 2. Load the dataset

If you are using Google Colab, upload `amazon.csv` to the session first or update `DATA_PATH` to the correct file path.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}\n"
        "Place amazon.csv in the notebook folder or update DATA_PATH."
    )

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

## 3. Understand the data

Inspect the field names, data types, descriptive statistics, and examples before cleaning. This helps avoid treating text categories as numeric measures.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

display(df.describe(include="all").T)

## 4. Clean column names and numeric fields

Prices and percentages may contain symbols such as `₹`, commas, or `%`. Convert these fields to numeric values so calculations and charts work correctly. The cleaning functions safely handle missing values and unexpected text.

In [ ]:
# Standardize column names while preserving readable snake_case.
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

def clean_numeric(series, remove_chars=()):
    """Convert currency/percentage-like text to numeric values."""
    result = series.astype("string").str.strip()
    for char in remove_chars:
        result = result.str.replace(char, "", regex=False)
    result = result.str.replace(",", "", regex=False)
    return pd.to_numeric(result, errors="coerce")

if "actual_price" in df.columns:
    df["actual_price"] = clean_numeric(df["actual_price"], remove_chars=("₹",))
if "discounted_price" in df.columns:
    df["discounted_price"] = clean_numeric(df["discounted_price"], remove_chars=("₹",))
if "discount_percentage" in df.columns:
    df["discount_percentage"] = clean_numeric(df["discount_percentage"], remove_chars=("%",))
if "rating_count" in df.columns:
    df["rating_count"] = clean_numeric(df["rating_count"])
if "rating" in df.columns:
    df["rating"] = clean_numeric(df["rating"])

print("Data types after cleaning:")
display(df.dtypes.astype(str).to_frame("dtype"))

## 5. Data-quality assessment

Measure missing values and duplicate rows before deciding what to remove. Missing review counts or ratings may have meaning, so this notebook reports data quality first and uses analysis-specific filtering later.

In [ ]:
duplicate_rows = int(df.duplicated().sum())
print(f"Duplicate rows: {duplicate_rows:,} ({duplicate_rows / max(len(df), 1):.2%})")

quality_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_percent", ascending=False)

display(quality_report)

missing_pct = quality_report.loc[quality_report["missing_count"] > 0, "missing_percent"]
if not missing_pct.empty:
    ax = missing_pct.sort_values().plot(
        kind="barh", figsize=(9, max(3, len(missing_pct) * 0.32))
    )
    ax.set_title("Missing Values by Column")
    ax.set_xlabel("Missing values (%)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values detected.")

## 6. Standardize product categories

Amazon datasets often store category paths separated by `|`. Create a main-category field for easier grouping, while keeping the original category field for reference.

In [ ]:
if "category" in df.columns:
    df["category"] = df["category"].astype("string").str.strip()
    df["main_category"] = df["category"].str.split("|").str[0].str.strip()
    print(f"Unique main categories: {df['main_category'].nunique(dropna=True)}")
    display(df[["category", "main_category"]].head())
else:
    print("The dataset does not contain a 'category' column.")

## 7. Create useful analysis metrics

Calculate discount amount and discount percentage from prices where possible. Compare the calculated percentage with the provided value to flag possible inconsistencies. These are checks, not proof that a record is incorrect.

In [ ]:
if {"actual_price", "discounted_price"}.issubset(df.columns):
    df["discount_amount"] = df["actual_price"] - df["discounted_price"]
    df["calculated_discount_pct"] = np.where(
        df["actual_price"] > 0,
        df["discount_amount"] / df["actual_price"] * 100,
        np.nan
    )
    display(df[["actual_price", "discounted_price", "discount_amount",
                "discount_percentage", "calculated_discount_pct"]].head())
else:
    print("Price metrics skipped because actual_price or discounted_price is missing.")

## 8. Price distribution

View actual and discounted prices separately. E-commerce prices are often skewed, so the histograms and boxplots can help identify the spread and unusually high values.

In [ ]:
price_cols = [c for c in ["actual_price", "discounted_price"] if c in df.columns]

for col in price_cols:
    values = df[col].dropna()
    if values.empty:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(values, bins=30, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {col.replace('_', ' ').title()}")
    axes[0].set_xlabel("Price (₹)")

    sns.boxplot(x=values, ax=axes[1])
    axes[1].set_title(f"Potential Outliers: {col.replace('_', ' ').title()}")
    axes[1].set_xlabel("Price (₹)")

    plt.tight_layout()
    plt.show()

    display(values.describe().to_frame(col))

## 9. Discount analysis

Compare the distribution of discounts and identify products with the largest advertised discounts. Verify extreme values against the original source before reporting them.

In [ ]:
if "discount_percentage" in df.columns:
    discount = df["discount_percentage"].dropna()
    if not discount.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(discount, bins=25, kde=True, ax=axes[0])
        axes[0].set_title("Distribution of Discount Percentage")
        axes[0].set_xlabel("Discount (%)")

        sns.boxplot(x=discount, ax=axes[1])
        axes[1].set_title("Discount Percentage: Potential Outliers")
        axes[1].set_xlabel("Discount (%)")
        plt.tight_layout()
        plt.show()

    product_cols = [c for c in ["product_name", "discount_percentage",
                                "actual_price", "discounted_price"] if c in df.columns]
    if "discount_percentage" in product_cols:
        top_discounts = df.sort_values("discount_percentage", ascending=False)[product_cols].head(10)
        display(top_discounts)
else:
    print("Discount analysis skipped: 'discount_percentage' is missing.")

## 10. Product category overview

Show listing counts by main category. Listing count is not the same as sales volume or revenue; this dataset should only be described as sales data if actual transaction data is available.

In [ ]:
if "main_category" in df.columns:
    category_counts = (
        df["main_category"].fillna("Unknown")
        .value_counts()
        .head(20)
        .sort_values()
    )
    ax = category_counts.plot(kind="barh", figsize=(10, max(5, len(category_counts) * 0.32)))
    ax.set_title("Top Main Categories by Number of Product Listings")
    ax.set_xlabel("Number of listings")
    ax.set_ylabel("Main category")
    plt.tight_layout()
    plt.show()
    display(category_counts.sort_values(ascending=False).to_frame("product_listings"))
else:
    print("Category analysis skipped: 'main_category' is missing.")

## 11. Category-level price, discount, and rating summary

Summarize product listings by category using count, average prices, average discount, and average rating. Include sample size when comparing ratings because categories can have very different numbers of listings.

In [ ]:
group_cols = ["main_category"] if "main_category" in df.columns else []
metrics = [c for c in ["actual_price", "discounted_price",
                       "discount_percentage", "rating", "rating_count"] if c in df.columns]

if group_cols and metrics:
    category_summary = df.groupby("main_category", dropna=False).agg(
        product_listings=("main_category", "size"),
        **{f"mean_{col}": (col, "mean") for col in metrics}
    ).sort_values("product_listings", ascending=False)

    display(category_summary.round(2))
else:
    print("Category summary skipped: category or metric columns are unavailable.")

## 12. Average rating by category

This chart compares average product ratings across categories. Use the listing count alongside the mean and avoid interpreting small groups as definitive evidence of quality.

In [ ]:
if "main_category" in df.columns and "rating" in df.columns:
    rating_summary = (
        df.groupby("main_category")["rating"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "average_rating", "count": "rated_listings"})
        .dropna(subset=["average_rating"])
        .sort_values("average_rating", ascending=False)
    )
    display(rating_summary.round(2))

    plot_data = rating_summary.sort_values("average_rating")
    ax = plot_data["average_rating"].plot(kind="barh", figsize=(10, max(5, len(plot_data) * 0.32)))
    ax.set_title("Average Product Rating by Main Category")
    ax.set_xlabel("Average rating")
    ax.set_ylabel("Main category")
    plt.tight_layout()
    plt.show()
else:
    print("Rating-by-category analysis skipped: required columns are missing.")

## 13. Review engagement by category

Sum review counts by category only if `rating_count` is available. Review counts represent engagement or recorded reviews, not sales. Categories with many listings may naturally accumulate more reviews, so consider the average review count per listing too.

In [ ]:
if "main_category" in df.columns and "rating_count" in df.columns:
    review_summary = (
        df.groupby("main_category")["rating_count"]
        .agg(total_reviews="sum", average_reviews_per_listing="mean", listing_count="count")
        .sort_values("total_reviews", ascending=False)
    )
    display(review_summary.round(2))

    top_reviews = review_summary.head(15).sort_values("total_reviews")
    ax = top_reviews["total_reviews"].plot(kind="barh", figsize=(10, 6))
    ax.set_title("Top Categories by Total Recorded Review Count")
    ax.set_xlabel("Total review count")
    ax.set_ylabel("Main category")
    plt.tight_layout()
    plt.show()
else:
    print("Review analysis skipped: category or rating_count is missing.")

## 14. Explore relationships between price, discount, rating, and reviews

Scatterplots can reveal patterns and outliers. With many records, a reproducible sample improves readability. These relationships are descriptive and do not establish causation.

In [ ]:
relationships = [
    ("actual_price", "rating", "Actual Price vs Rating"),
    ("discount_percentage", "rating", "Discount Percentage vs Rating"),
    ("discounted_price", "actual_price", "Discounted Price vs Actual Price"),
    ("rating_count", "rating", "Review Count vs Rating"),
]

for x_col, y_col, chart_title in relationships:
    if x_col in df.columns and y_col in df.columns:
        plot_df = df[[x_col, y_col]].dropna()
        if plot_df.empty:
            continue
        if len(plot_df) > 5000:
            plot_df = plot_df.sample(5000, random_state=42)

        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=plot_df, x=x_col, y=y_col, alpha=0.4)
        plt.title(chart_title)
        plt.tight_layout()
        plt.show()

## 15. Correlation analysis

Use only numeric fields. Correlation can help identify linear associations, but category labels should not be label-encoded just to include them in a correlation matrix; arbitrary category codes can create misleading results.

In [ ]:
numeric_candidates = [
    c for c in ["actual_price", "discounted_price", "discount_amount",
                "discount_percentage", "calculated_discount_pct",
                "rating", "rating_count"]
    if c in df.columns
]
numeric_df = df[numeric_candidates].apply(pd.to_numeric, errors="coerce")

if len(numeric_candidates) >= 2:
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))

    plt.figure(figsize=(9, 7))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
                cmap="vlag", center=0, square=True, linewidths=0.5)
    plt.title("Correlation Between Numeric Product Fields")
    plt.tight_layout()
    plt.show()
    display(corr.round(3))
else:
    print("At least two numeric fields are required for correlation analysis.")

## 16. Executive summary metrics

Generate concise metrics from the dataset. Use the results below when writing your project findings, and avoid inventing sales or revenue conclusions because product-listing data is not necessarily transaction data.

In [ ]:
print("AMAZON PRODUCT ANALYSIS — SNAPSHOT")
print("=" * 42)
print(f"Product listing rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Duplicate rows: {int(df.duplicated().sum()):,}")

if "main_category" in df.columns:
    print(f"Unique main categories: {df['main_category'].nunique(dropna=True):,}")
    if df["main_category"].notna().any():
        print(f"Category with most listings: {df['main_category'].value_counts().idxmax()}")

for col, label in [
    ("actual_price", "Average actual price"),
    ("discounted_price", "Average discounted price"),
    ("discount_percentage", "Average listed discount"),
    ("rating", "Average product rating"),
    ("rating_count", "Average recorded review count")
]:
    if col in df.columns:
        print(f"{label}: {df[col].mean():,.2f}")

if {"actual_price", "discounted_price"}.issubset(df.columns):
    print(f"Products with positive discount amount: {(df['discount_amount'] > 0).sum():,}")

## 17. Conclusions and business recommendations

After running the notebook, complete these statements with actual results:

1. **Category mix:** The category with the most product listings was ______, with ______ listings.
2. **Pricing:** The average actual price was ______ and the average discounted price was ______.
3. **Discounts:** The average listed discount was ______%; verify whether this aligns with the calculated discount.
4. **Ratings:** The category with the highest average rating was ______, based on ______ rated listings.
5. **Review engagement:** The category with the highest recorded review count was ______. This indicates engagement, not necessarily sales.
6. **Data quality:** The most important issue found was ______, and the recommended treatment is ______.

### Limitations
- Product-listing data does not necessarily contain orders, units sold, or revenue.
- Ratings and review counts can be influenced by listing age, product popularity, and category size.
- Correlation does not imply causation.
- Verify source, currency, collection date, and field definitions before presenting findings.

## 18. Portfolio-ready project description

**Amazon Product & Pricing Analysis | Python**

Performed exploratory data analysis on Amazon product listings using Python, Pandas, NumPy, Matplotlib, and Seaborn. Cleaned price, discount, rating, and review-count fields; assessed missing values and duplicates; compared pricing and ratings across product categories; and visualized relationships between discounts, ratings, and review engagement. Generated category-level summaries and an executive metrics snapshot to support data-driven product analysis.

**Before publishing:** Replace this generic description with 2–3 quantified findings from your executed notebook.